In [1]:
!pip install xgboost lightgbm scikit-learn pandas numpy joblib matplotlib --quiet

import pandas as pd
import numpy as np
import gc
import joblib
import warnings
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from lightgbm import LGBMClassifier

print("Imports successful")

Imports successful


In [2]:
data         = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/splits.pkl")
X_train      = data["X_train"]
X_val        = data["X_val"]
X_test       = data["X_test"]
y_train      = data["y_train"]
y_val        = data["y_val"]
y_test       = data["y_test"]
del data
gc.collect()

le           = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/label_encoder.pkl")
feature_cols = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/feature_cols.pkl")
le_xgb       = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/le_xgb.pkl")
rf_model     = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/rf_model.pkl")
xgb_model    = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/xgb_model.pkl")
svm_model    = joblib.load("/kaggle/input/datasets/pes2ug23cs197/ensemble-part2/svm_model.pkl")

num_classes  = len(le.classes_)

print("All files loaded")
print(f"Train    : {X_train.shape[0]} rows")
print(f"Val      : {X_val.shape[0]} rows")
print(f"Test     : {X_test.shape[0]} rows")
print(f"Features : {X_train.shape[1]}")
print(f"Classes  : {num_classes}")

All files loaded
Train    : 34999 rows
Val      : 7500 rows
Test     : 7500 rows
Features : 1756
Classes  : 1117


In [3]:
print("Generating val and test meta-features...\n")

meta_val = np.hstack([
    rf_model.predict_proba(X_val),
    xgb_model.predict_proba(X_val),
    svm_model.predict_proba(X_val),
]).astype(np.float32)

meta_test = np.hstack([
    rf_model.predict_proba(X_test),
    xgb_model.predict_proba(X_test),
    svm_model.predict_proba(X_test),
]).astype(np.float32)

print(f"meta_val  shape : {meta_val.shape}")
print(f"meta_test shape : {meta_test.shape}")

print("\n--- Base Model Val Accuracy After Full Retrain ---")

# Random Forest
rf_proba = rf_model.predict_proba(X_val)
rf_preds = rf_model.classes_[np.argmax(rf_proba, axis=1)]
rf_acc   = accuracy_score(y_val, rf_preds)
print(f"  RF          : {rf_acc*100:.2f}%")

# XGBoost
xgb_proba = xgb_model.predict_proba(X_val)
xgb_preds = le_xgb.inverse_transform(np.argmax(xgb_proba, axis=1))
xgb_acc   = accuracy_score(y_val, xgb_preds)
print(f"  XGBoost     : {xgb_acc*100:.2f}%")

# SVM
svm_proba = svm_model.predict_proba(X_val)
svm_preds = svm_model.classes_[np.argmax(svm_proba, axis=1)]
svm_acc   = accuracy_score(y_val, svm_preds)
print(f"  SVM         : {svm_acc*100:.2f}%")

Generating val and test meta-features...

meta_val  shape : (7500, 1449)
meta_test shape : (7500, 1449)

--- Base Model Val Accuracy After Full Retrain ---
  RF          : 81.81%
  XGBoost     : 73.33%
  SVM         : 81.85%


In [4]:
print("Rebuilding meta_train from full base models...\n")

meta_train = np.hstack([
    rf_model.predict_proba(X_train),
    xgb_model.predict_proba(X_train),
    svm_model.predict_proba(X_train),
]).astype(np.float32)

print(f"meta_train shape : {meta_train.shape}")
print(f" = {X_train.shape[0]} samples × "
      f"({num_classes} classes × 3 models = {num_classes*3} meta-features)")

Rebuilding meta_train from full base models...

meta_train shape : (34999, 1449)
 = 34999 samples × (1117 classes × 3 models = 3351 meta-features)


In [5]:
meta_learner = LGBMClassifier(
    n_estimators      = 300,       
    learning_rate     = 0.03,       
    max_depth         = -1,         
    num_leaves        = 128,        
    min_child_samples = 20,         
    subsample         = 0.9,        
    colsample_bytree  = 0.9,        
    class_weight      = "balanced",
    random_state      = 42,
    n_jobs            = -1,
    verbose           = -1
)

print("Training LightGBM ...")

meta_learner.fit(
    meta_train, y_train,
    eval_set             = [(meta_val, y_val)],
    callbacks            = [
        __import__("lightgbm").early_stopping(stopping_rounds=30, verbose=False),
        __import__("lightgbm").log_evaluation(period=50)
    ]
)

meta_val_pred = meta_learner.predict(meta_val)
meta_val_acc  = accuracy_score(y_val, meta_val_pred)
print(f"\nLightGBM — Validation Accuracy: {meta_val_acc*100:.2f}%")

Training LightGBM ...

LightGBM — Validation Accuracy: 74.84%


In [6]:
print("\n--- Accuracy Comparison  ---\n")

xgb_val_preds = le_xgb.inverse_transform(xgb_model.predict(X_val))

results = {
    "Random Forest"       : accuracy_score(y_val, rf_model.predict(X_val)),
    "XGBoost"             : accuracy_score(y_val, xgb_val_preds),             
    "SVM"                 : accuracy_score(y_val, svm_model.predict(X_val)),
    "Stacking Ensemble"   : meta_val_acc,
}

for name, acc in results.items():
    bar = "█" * int(acc * 40)
    print(f"  {name:<25} {acc*100:.2f}%  {bar}")

best_base = max(
    results["Random Forest"],
    results["XGBoost"],
    results["SVM"]
)
lift = meta_val_acc - best_base
print(f"\n  Ensemble lift over best base model: +{lift*100:.2f}%")


--- Accuracy Comparison  ---

  Random Forest             81.81%  ████████████████████████████████
  XGBoost                   73.33%  █████████████████████████████
  SVM                       81.85%  ████████████████████████████████
  Stacking Ensemble         74.84%  █████████████████████████████

  Ensemble lift over best base model: +-7.01%


In [7]:
print("\n--- Final Test Set Evaluation ---")

meta_test_pred        = meta_learner.predict(meta_test)
meta_test_pred_proba  = meta_learner.predict_proba(meta_test)
test_acc              = accuracy_score(y_test, meta_test_pred)

meta_train_pred       = meta_learner.predict(meta_train)
train_acc             = accuracy_score(y_train, meta_train_pred)
gap                   = train_acc - test_acc

del meta_train_pred
gc.collect()

print(f"  Train Accuracy   : {train_acc*100:.2f}%")
print(f"  Val   Accuracy   : {meta_val_acc*100:.2f}%")
print(f"  Test  Accuracy   : {test_acc*100:.2f}%")
print(f"  Gap (Train-Test) : {gap*100:.2f}%")

if gap < 0.05:
    print("Excellent — no significant overfitting")
elif gap < 0.10:
    print("Acceptable — mild overfit, normal for this domain")
else:
    print("Gap > 10% — consider reducing num_leaves or increasing min_child_samples")


--- Final Test Set Evaluation ---
  Train Accuracy   : 87.58%
  Val   Accuracy   : 74.84%
  Test  Accuracy   : 74.87%
  Gap (Train-Test) : 12.72%
Gap > 10% — consider reducing num_leaves or increasing min_child_samples


In [8]:
final_bundle = {
    "meta_learner"  : meta_learner,
    "rf_model"      : rf_model,
    "xgb_model"     : xgb_model,
    "svm_model"     : svm_model,
    "label_encoder" : le,
    "le_xgb"        : le_xgb,          
    "feature_cols"  : feature_cols,
}

joblib.dump(final_bundle, "/kaggle/working/final_model.pkl")
print("final_model.pkl saved")

final_model.pkl saved
